<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding Duplicates Lab**


Estimated time needed: **30** minutes


## Introduction


Data wrangling is a critical step in preparing datasets for analysis, and handling duplicates plays a key role in ensuring data accuracy. In this lab, you will focus on identifying and removing duplicate entries from your dataset. 


## Objectives


In this lab, you will perform the following:


1. Identify duplicate rows in the dataset and analyze their characteristics.
2. Visualize the distribution of duplicates based on key attributes.
3. Remove duplicate values strategically based on specific criteria.
4. Outline the process of verifying and documenting duplicate removal.


## Hands on Lab


Install the needed library


In [ ]:
# The original `!pip install` magic commands only run in Jupyter environments.
# They are kept below as comments. Use subprocess instead so the packages are
# installed with the same Python interpreter that is selected in VS Code.
# !pip install pandas
# !pip install matplotlib
import importlib.util
import subprocess
import sys

def install(package, import_name=None):
    """Quietly pip-install a package if it is not already available."""
    import_name = import_name or package
    try:
        spec = importlib.util.find_spec(import_name)
    except (ImportError, ModuleNotFoundError, ValueError):
        spec = None
    if spec is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])

install('pandas', 'pandas')
install('matplotlib', 'matplotlib')

Import pandas module


In [ ]:
import pandas as pd


Import matplotlib


In [ ]:
import matplotlib.pyplot as plt


## **Load the dataset into a dataframe**


<h2>Read Data</h2>
<p>
We utilize the <code>pandas.read_csv()</code> function for reading CSV files. However, in this version of the lab, which operates on JupyterLite, the dataset needs to be downloaded to the interface using the provided code below.
</p>


In [ ]:
# Load the dataset directly from the URL
file_path = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VYPrOu0Vs3I0hKLLjiPGrA/survey-data-with-duplicate.csv"
df = pd.read_csv(file_path)

# Display the first few rows
print(df.head())

Load the data into a pandas dataframe:



Note: If you are working on a local Jupyter environment, you can use the URL directly in the pandas.read_csv() function as shown below:



In [ ]:
# df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv")


## Identify and Analyze Duplicates


### Task 1: Identify Duplicate Rows
1. Count the number of duplicate rows in the dataset.
3. Display the first few duplicate rows to understand their structure.


In [ ]:
## Write your code here
# 1) Count the number of duplicate rows in the dataset
num_duplicates = df.duplicated().sum()          # rows that repeat a previous row
all_copies = df.duplicated(keep=False).sum()    # duplicated rows plus their copies
print("Number of duplicate rows:", num_duplicates)
print("Rows involved in duplication (all copies):", all_copies)

# 2) Display the first few duplicate rows to understand their structure
df[df.duplicated(keep=False)].sort_values('ResponseId').head(10)

### Task 2: Analysis of Duplicate Response Patterns
1. Identify duplicate response patterns based on selected columns such as MainBranch, Employment, and RemoteWork.
2. Clarify that these represent multiple respondents with identical answers rather than duplicate records. Analyse which other columns frequently share identical values within these response-pattern groups.
   


In [ ]:
## Write your code here
pattern_cols = ['MainBranch', 'Employment', 'RemoteWork']

# Rows that share their MainBranch/Employment/RemoteWork pattern with other respondents
# (keep=False marks every member of a shared-pattern group)
pattern_dupes = df[df.duplicated(subset=pattern_cols, keep=False)]

# How many respondents share each identical response pattern
pattern_counts = pattern_dupes.groupby(pattern_cols).size().sort_values(ascending=False)
print("Respondents sharing a response pattern:", len(pattern_dupes))
print("Number of unique response patterns:", len(pattern_counts))
print("\nTop 10 most common response patterns:")
print(pattern_counts.head(10))

# Which other columns frequently share identical values inside these groups:
# count how many distinct values each column takes within every pattern group
nunique_in_group = pattern_dupes.groupby(pattern_cols).nunique(dropna=False)
share_identical = (nunique_in_group == 1).mean().sort_values(ascending=False)
print("\nShare of pattern groups in which a column has a single identical value (top 10):")
print(share_identical.head(10).round(3))

### Task 3: Visualize Shared Response Patterns
1. Create visualizations to show the distribution of shared response patterns across different categories.
2. Use bar charts or pie charts to represent the distribution of respondents who share identical values for MainBranch, Employment, and RemoteWork, grouped by Country and Employment.


In [ ]:
## Write your code here
pattern_cols = ['MainBranch', 'Employment', 'RemoteWork']
pattern_dupes = df[df.duplicated(subset=pattern_cols, keep=False)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart 1: distribution of shared response patterns by Country (top 10)
pattern_dupes['Country'].value_counts().head(10).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Respondents sharing identical MainBranch/Employment/RemoteWork - by Country')
axes[0].set_ylabel('Number of respondents')
axes[0].tick_params(axis='x', rotation=45)

# Bar chart 2: distribution of shared response patterns by Employment (top 10)
pattern_dupes['Employment'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Respondents sharing identical MainBranch/Employment/RemoteWork - by Employment')
axes[1].set_ylabel('Number of respondents')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Task 4: Evaluate Duplicate Handling Strategy
1. Analyse the dataset to determine which column(s) define record uniqueness.
2. Assess whether removing rows based on a subset of columns (rather than complete row duplication) is appropriate.
Justify your decision with reference to the structure and purpose of the dataset.


<!-- ## Write your analysis here -->

**Which column(s) define record uniqueness?** `ResponseId` is the unique record identifier: the dataset has 65,457 rows and 65,437 unique `ResponseId` values, and the only repeated IDs belong to the 20 exact full-row duplicates. A record is therefore unique when its `ResponseId` is new; in practice, `df.duplicated()` (full-row comparison) identifies the duplicates, with `ResponseId` confirming them.

**Is removing rows on a subset of columns appropriate?** No - not for `MainBranch`, `Employment`, and `RemoteWork`. Those three columns form coarse categorical attributes: 65,290 rows fall into only 319 shared-pattern groups (the largest single group holds 15,288 respondents). Dropping "duplicates" on that subset would collapse the dataset to ~319 rows and delete almost every respondent.

**Justification.** Each row is one survey respondent, and respondents with identical answers are different people, not duplicated records (e.g., thousands of full-time hybrid developers in the USA, Germany, and India). Only the 20 rows whose entire 114-column record is repeated are true duplicates, so the correct strategy is `df.drop_duplicates()` on the full row (or on `ResponseId`), never on a small answer subset.

## Verify and Document Duplicate Removal Process


### Task 5: Documentation
1. Document the process of identifying and removing duplicates.


<!-- # Write your explanation here -->

**Process for identifying and removing duplicates:**

1. **Load** the dataset (`survey-data-with-duplicate.csv`) into a dataframe.
2. **Count duplicates** with `df.duplicated().sum()` -> 20 duplicate rows.
3. **Inspect them** with `df.duplicated(keep=False)` -> 40 rows (the duplicates plus their originals); sort by `ResponseId` to see that every copy is byte-for-byte identical across all 114 columns.
4. **Verify the key**: `df['ResponseId'].nunique()` -> 65,437, so the only repeated `ResponseId` values are those duplicated records.
5. **Remove** them with `df = df.drop_duplicates()` (keeps the first copy of each record).
6. **Confirm**: the shape goes from (65457, 114) to (65437, 114) and `df.duplicated().sum()` returns 0.

The cleaned dataframe is the analysis-ready version; the original file is left untouched on disk.

2. Explain the reasoning behind selecting specific columns for identifying and removing duplicates.


<!-- # Write your explanation here -->

**Why full-row duplication (and `ResponseId`), not a subset of columns:**

- The purpose of the dataset is one row per survey respondent; uniqueness is defined by the respondent's identifier `ResponseId`, which matches the full-row duplicate count exactly (20).
- Columns such as `MainBranch`, `Employment`, and `RemoteWork` are shared attributes by design - with only 3 values each, 99.4% of respondents (65,290 of 65,457) coincide with at least one other respondent, so they cannot define uniqueness.
- Removing on such a subset would destroy real, distinct responses (e.g., it would keep only 15,288 of the hybrid full-time developers in the largest group) and bias every later statistic, especially salary and country analyses.
- Columns like `LanguageHaveWorkedWith` or `LearnCode` vary almost per person (they are identical in only ~1% of pattern groups), so they are also unsuitable as dedup keys - any subset rule would either delete too much or too little.
- Therefore the defensible rule is: drop only exact full-row duplicates (equivalently, repeated `ResponseId`), which preserves all 65,437 distinct respondents.

### Summary and Next Steps
**In this lab, you focused on identifying and analyzing duplicate rows within the dataset.**

- You employed various techniques to explore the nature of duplicates and applied strategic methods for their removal.
- For additional analysis, consider investigating the impact of duplicates on specific analyses and how their removal affects the results.
- This version of the lab is more focused on duplicate analysis and handling, providing a structured approach to deal with duplicates in a dataset effectively.


<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-11- 05|1.3|Madhusudhan Moole|Updated lab|
|2024-10-28|1.2|Madhusudhan Moole|Updated lab|
|2024-09-24|1.1|Madhusudhan Moole|Updated lab|
|2024-09-23|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
